# 04 LLM-as-a-Judge Evaluation: Qwen3-32B, Single-Summary Only, All Patients

This notebook evaluates baseline, RAG, RAG2 multi-query, and RAG3 matched-budget BHC summaries using **single-summary LLM-as-a-judge scoring only**. It removes pairwise comparison, uses `Qwen/Qwen3-32B`, supports checkpointing, and evaluates all available patients.

In [ ]:
# 1. Install packages
!pip install -q -U transformers accelerate bitsandbytes pandas pyarrow tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 106.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 166.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 56.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.2 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.2 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but 

In [ ]:
# 2. Imports
import os, re, json, time
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [ ]:
# 3. Mount Google Drive and set paths
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = "/content/drive/MyDrive/BMI702 Project/Data"  # edit if needed
print("DATA_DIR:", DATA_DIR)
print("Exists:", os.path.exists(DATA_DIR))
if not os.path.exists(DATA_DIR):
    raise FileNotFoundError(DATA_DIR)

FILES = {
    "baseline_results": "baseline_results.parquet",
    "rag_results": "rag_results.parquet",
    "rag2_results": "rag2_multi_query_results.parquet",
    "rag3_results": "rag3_matched_budget_results.parquet",
    "human_bhcs": "human_bhcs.parquet",
    "notes": "notes.parquet",
    "retrieved_context": "retrieved_context.parquet",  # optional
}
file_paths = {k: os.path.join(DATA_DIR, v) for k, v in FILES.items()}
for k, p in file_paths.items():
    print(f"{k:20s}", os.path.exists(p), p)

required = ["baseline_results", "rag_results", "rag2_results", "rag3_results", "human_bhcs", "notes"]
missing = [k for k in required if not os.path.exists(file_paths[k])]
if missing:
    raise FileNotFoundError(f"Missing required files: {missing}")

Mounted at /content/drive
DATA_DIR: /content/drive/MyDrive/BMI702 Project/Data
Exists: True
baseline_results     True /content/drive/MyDrive/BMI702 Project/Data/baseline_results.parquet
rag_results          True /content/drive/MyDrive/BMI702 Project/Data/rag_results.parquet
rag2_results         True /content/drive/MyDrive/BMI702 Project/Data/rag2_multi_query_results.parquet
rag3_results         True /content/drive/MyDrive/BMI702 Project/Data/rag3_matched_budget_results.parquet
human_bhcs           True /content/drive/MyDrive/BMI702 Project/Data/human_bhcs.parquet
notes                True /content/drive/MyDrive/BMI702 Project/Data/notes.parquet
retrieved_context    False /content/drive/MyDrive/BMI702 Project/Data/retrieved_context.parquet


In [ ]:
# 4. Read files
def read_required(key):
    df = pd.read_parquet(file_paths[key])
    print(f"Loaded {key}: {df.shape}")
    return df

def read_optional(key):
    p = file_paths[key]
    if os.path.exists(p):
        df = pd.read_parquet(p)
        print(f"Loaded optional {key}: {df.shape}")
        return df
    print(f"Optional file not found: {key}")
    return None

baseline_results = read_required("baseline_results")
rag_results = read_required("rag_results")
rag2_results = read_required("rag2_results")
rag3_results = read_required("rag3_results")
human_bhcs = read_required("human_bhcs")
notes = read_required("notes")
retrieved_context = read_optional("retrieved_context")

Loaded baseline_results: (100, 7)
Loaded rag_results: (100, 11)
Loaded rag2_results: (100, 11)
Loaded rag3_results: (100, 11)
Loaded human_bhcs: (100, 5)
Loaded notes: (4787, 13)
Optional file not found: retrieved_context


In [ ]:
# 5. Inspect columns
for name, df in [
    ("baseline_results", baseline_results),
    ("rag_results", rag_results),
    ("rag2_results", rag2_results),
    ("rag3_results", rag3_results),
    ("human_bhcs", human_bhcs),
    ("notes", notes),
    ("retrieved_context", retrieved_context),
]:
    if df is not None:
      print("" + "="*80)
      print(name, df.shape)
      print(df.columns.tolist())
      display(df.head(2))

baseline_results (100, 7)
['subject_id', 'input_tokens', 'was_truncated', 'generated_bhc', 'generated_tokens', 'generation_time_sec', 'human_bhc']


,subject_id,input_tokens,was_truncated,generated_bhc,generated_tokens,generation_time_sec,human_bhc
0,1084,16193,False,The patient is a 61-year-old male with a histo...,486,41.5,Pt is a 61 y.o male with h.o prostate ca with ...
1,4954,57192,True,**Hospital Course Summary:**\n\nThe patient is...,1025,87.5,"64yo woman with multiple myeloma, s/p allogene..."


rag_results (100, 11)
['subject_id', 'full_note_tokens', 'retrieved_tokens', 'input_tokens', 'compression_ratio', 'was_truncated_baseline', 'generated_bhc', 'generated_tokens', 'generation_time_sec', 'human_bhc', 'n_chunks_retrieved']


,subject_id,full_note_tokens,retrieved_tokens,input_tokens,compression_ratio,was_truncated_baseline,generated_bhc,generated_tokens,generation_time_sec,human_bhc,n_chunks_retrieved
0,1084,15988,10840,11705,0.678009,False,"The patient, a 61-year-old male with a history...",418,36.8,Pt is a 61 y.o male with h.o prostate ca with ...,40
1,4954,56289,10658,11528,0.189344,True,"The patient, a 64-year-old individual with a h...",656,56.6,"64yo woman with multiple myeloma, s/p allogene...",40


rag2_results (100, 11)
['subject_id', 'full_note_tokens', 'retrieved_tokens', 'input_tokens', 'compression_ratio', 'was_truncated_baseline', 'generated_bhc', 'generated_tokens', 'generation_time_sec', 'human_bhc', 'n_chunks_retrieved']


...

,subject_id,full_note_tokens,retrieved_tokens,input_tokens,compression_ratio,was_truncated_baseline,generated_bhc,generated_tokens,generation_time_sec,human_bhc,n_chunks_retrieved
0,1084,15988,11813,12755,0.738867,False,"The patient, a 61-year-old male with a history...",416,36.2,Pt is a 61 y.o male with h.o prostate ca with ...,44
1,4954,56289,27876,29948,0.495230,True,"**Brief Hospital Course:**\n\nThe patient, a 6...",445,41.1,"64yo woman with multiple myeloma, s/p allogene...",102


human_bhcs (100, 5)
['subject_id', 'hadm_id', 'brief_hospital_course', 'bhc_char_length', 'bhc_approx_tokens']


,subject_id,hadm_id,brief_hospital_course,bhc_char_length,bhc_approx_tokens
0,1084,194111,Pt is a 61 y.o male with h.o prostate ca with ...,2047,511
1,4954,158018,"64yo woman with multiple myeloma, s/p allogene...",2532,633


notes (4787, 13)
['ROW_ID', 'SUBJECT_ID', 'HADM_ID', 'CHARTDATE', 'CHARTTIME', 'STORETIME', 'CATEGORY', 'DESCRIPTION', 'CGID', 'ISERROR', 'TEXT', 'char_length', 'approx_tokens']


,ROW_ID,SUBJECT_ID,HADM_ID,CHARTDATE,CHARTTIME,STORETIME,CATEGORY,DESCRIPTION,CGID,ISERROR,TEXT,char_length,approx_tokens
0,384044,1084,194111.0,2198-08-02,2198-08-02 19:30:00,2198-08-02 19:38:21,Physician,Physician Resident Admission Note,14770.0,NaN,Chief Complaint: Primary Care Physician: [**N...,8366,2091
1,384047,1084,194111.0,2198-08-02,2198-08-02 19:30:00,2198-08-02 21:00:44,Physician,Physician Resident Admission Note,17419.0,NaN,Chief Complaint: Primary Care Physician: [**N...,9912,2478


In [ ]:
# 6. Column detection utilities
ID_CANDIDATES = ["subject_id", "SUBJECT_ID", "patient_id", "PATIENT_ID", "hadm_id", "HADM_ID", "icustay_id", "ICUSTAY_ID", "_pid"]
SUMMARY_CANDIDATES = ["generated_bhc", "generated_summary", "summary", "bhc", "BHC", "baseline_bhc", "baseline_summary", "rag_bhc", "rag_summary", "rag2_bhc", "rag2_summary", "rag2_multi_query_summary", "rag3_bhc", "rag3_summary", "rag3_matched_budget_summary", "text", "TEXT"]
REFERENCE_CANDIDATES = ["human_bhc", "human_reference", "brief_hospital_course", "BHC", "bhc", "reference_bhc", "reference_summary", "text", "TEXT"]
NOTE_TEXT_CANDIDATES = ["TEXT", "text", "note_text", "chunk_text", "context", "retrieved_text"]
NOTE_CATEGORY_CANDIDATES = ["CATEGORY", "category", "note_type", "DESCRIPTION", "description"]
NOTE_DATE_CANDIDATES = ["CHARTDATE", "chartdate", "CHARTTIME", "charttime", "date"]

def find_col(df, candidates, required=True, df_name="dataframe"):
    if df is None:
        if required: raise ValueError(f"{df_name} is None")
        return None
    cols = list(df.columns)
    for c in candidates:
        if c in cols: return c
    lower_map = {str(col).lower(): col for col in cols}
    for c in candidates:
        if str(c).lower() in lower_map: return lower_map[str(c).lower()]
    for c in candidates:
        c_lower = str(c).lower()
        for col in cols:
            if c_lower in str(col).lower(): return col
    if required:
        raise ValueError(f"Cannot find columns in {df_name}. Candidates={candidates}. Available={cols}")
    return None

def standardize_result_df(df, id_col, text_col, system_name):
    out = df[[id_col, text_col]].copy()
    out["_pid"] = out[id_col].astype(str)
    out[f"{system_name}_summary"] = out[text_col].fillna("").astype(str)
    out = out[["_pid", f"{system_name}_summary"]]
    return out.drop_duplicates(subset=["_pid"], keep="first")

In [ ]:
# 7. Standardize baseline/RAG/RAG2/RAG3/human reference and merge
BASELINE_ID_COL = find_col(baseline_results, ID_CANDIDATES, df_name="baseline_results")
RAG_ID_COL = find_col(rag_results, ID_CANDIDATES, df_name="rag_results")
RAG2_ID_COL = find_col(rag2_results, ID_CANDIDATES, df_name="rag2_results")
RAG3_ID_COL = find_col(rag3_results, ID_CANDIDATES, df_name="rag3_results")
REF_ID_COL = find_col(human_bhcs, ID_CANDIDATES, df_name="human_bhcs")

BASELINE_TEXT_COL = find_col(baseline_results, SUMMARY_CANDIDATES, df_name="baseline_results")
RAG_TEXT_COL = find_col(rag_results, SUMMARY_CANDIDATES, df_name="rag_results")
RAG2_TEXT_COL = find_col(rag2_results, SUMMARY_CANDIDATES, df_name="rag2_results")
RAG3_TEXT_COL = find_col(rag3_results, SUMMARY_CANDIDATES, df_name="rag3_results")
REF_TEXT_COL = find_col(human_bhcs, REFERENCE_CANDIDATES, df_name="human_bhcs")

print("Using columns:")
print("BASELINE:", BASELINE_ID_COL, BASELINE_TEXT_COL)
print("RAG:", RAG_ID_COL, RAG_TEXT_COL)
print("RAG2:", RAG2_ID_COL, RAG2_TEXT_COL)
print("RAG3:", RAG3_ID_COL, RAG3_TEXT_COL)
print("REFERENCE:", REF_ID_COL, REF_TEXT_COL)

baseline_df = standardize_result_df(baseline_results, BASELINE_ID_COL, BASELINE_TEXT_COL, "baseline")
rag_df = standardize_result_df(rag_results, RAG_ID_COL, RAG_TEXT_COL, "rag")
rag2_df = standardize_result_df(rag2_results, RAG2_ID_COL, RAG2_TEXT_COL, "rag2_multi_query")
rag3_df = standardize_result_df(rag3_results, RAG3_ID_COL, RAG3_TEXT_COL, "rag3_matched_budget")

ref_df = human_bhcs[[REF_ID_COL, REF_TEXT_COL]].copy()
ref_df["_pid"] = ref_df[REF_ID_COL].astype(str)
ref_df["human_reference"] = ref_df[REF_TEXT_COL].fillna("").astype(str)
ref_df = ref_df[["_pid", "human_reference"]].drop_duplicates(subset=["_pid"], keep="first")

eval_df = (baseline_df
           .merge(rag_df, on="_pid", how="left")
           .merge(rag2_df, on="_pid", how="left")
           .merge(rag3_df, on="_pid", how="left")
           .merge(ref_df, on="_pid", how="left"))

for col in ["rag_summary", "rag2_multi_query_summary", "rag3_matched_budget_summary", "human_reference"]:
    eval_df[col] = eval_df[col].fillna("").astype(str)

print("eval_df:", eval_df.shape)
display(eval_df.head())

Using columns:
BASELINE: subject_id generated_bhc
RAG: subject_id generated_bhc
RAG2: subject_id generated_bhc
RAG3: subject_id generated_bhc
REFERENCE: subject_id brief_hospital_course
eval_df: (100, 6)


,_pid,baseline_summary,rag_summary,rag2_multi_query_summary,rag3_matched_budget_summary,human_reference
0,1084,The patient is a 61-year-old male with a histo...,"The patient, a 61-year-old male with a history...","The patient, a 61-year-old male with a history...","The patient, a 61-year-old male with a history...",Pt is a 61 y.o male with h.o prostate ca with ...
1,4954,**Hospital Course Summary:**\n\nThe patient is...,"The patient, a 64-year-old individual with a h...",The patient is a 63-year-old woman with a hist...,"**Brief Hospital Course:**\n\nThe patient, a 6...","64yo woman with multiple myeloma, s/p allogene..."
2,5954,"The patient, a 56-year-old male with a history...","The patient, a 56-year-old male with a history...",Title: Pacemaker Malfunction and Revision\n\nT...,Title: Pacemaker Malfunction and Revision\n\nA...,Mr. [**Known lastname 46**] is a 56 year-old m...
3,6214,**Hospital Course Summary:**\n\nThe patient is...,"The patient, a 73-year-old male, was admitted ...","The patient, a 73-year-old male, was transferr...",The patient is a 73-year-old male with a histo...,The patient was transferred from an OSH. He ar...
4,8501,The patient is a 25-year-old woman with a hist...,The patient is a 25-year-old female with a his...,The patient is a 25-year-old female with a his...,"The patient, a 25-year-old female with a histo...",Ms. [**Known lastname **] is a 25yo female wit...


In [ ]:
# =========================
# 8. Build clinical_context
# =========================

MAX_CONTEXT_CHARS_PER_PATIENT = 24000
MAX_NOTES_PER_PATIENT = 40
MAX_CHARS_PER_NOTE = 1200


def build_context_from_retrieved_context(df, eval_pids):
    id_col = find_col(df, ID_CANDIDATES, df_name="retrieved_context")
    text_col = find_col(df, NOTE_TEXT_CANDIDATES, df_name="retrieved_context")
    cat_col = find_col(
        df,
        NOTE_CATEGORY_CANDIDATES,
        required=False,
        df_name="retrieved_context"
    )
    date_col = find_col(
        df,
        NOTE_DATE_CANDIDATES,
        required=False,
        df_name="retrieved_context"
    )

    ctx = df.copy()
    ctx["_pid"] = ctx[id_col].astype(str)
    ctx = ctx[ctx["_pid"].isin(eval_pids)].copy()

    def fmt(row):
        cat = str(row[cat_col]) if cat_col else "retrieved chunk"
        date = str(row[date_col]) if date_col else "unknown date"
        text = str(row[text_col])[:MAX_CHARS_PER_NOTE]
        return f"[{cat} — {date}]\n{text}\n"

    ctx["piece"] = ctx.apply(fmt, axis=1)

    context_df = (
        ctx.groupby("_pid")["piece"]
        .apply(lambda x: "\n".join(list(x))[:MAX_CONTEXT_CHARS_PER_PATIENT])
        .reset_index()
        .rename(columns={"piece": "clinical_context"})
    )

    return context_df


def build_context_from_notes(df, eval_pids):
    id_col = find_col(df, ID_CANDIDATES, df_name="notes")
    text_col = find_col(df, NOTE_TEXT_CANDIDATES, df_name="notes")
    cat_col = find_col(
        df,
        NOTE_CATEGORY_CANDIDATES,
        required=False,
        df_name="notes"
    )
    date_col = find_col(
        df,
        NOTE_DATE_CANDIDATES,
        required=False,
        df_name="notes"
    )

    ctx = df.copy()
    ctx["_pid"] = ctx[id_col].astype(str)
    ctx = ctx[ctx["_pid"].isin(eval_pids)].copy()

    if date_col:
        ctx = ctx.sort_values(["_pid", date_col])

    def fmt(row):
        cat = str(row[cat_col]) if cat_col else "clinical note"
        date = str(row[date_col]) if date_col else "unknown date"
        text = str(row[text_col])[:MAX_CHARS_PER_NOTE]
        return f"[{cat} — {date}]\n{text}\n"

    ctx["piece"] = ctx.apply(fmt, axis=1)

    def combine(pieces):
        return "\n".join(list(pieces)[:MAX_NOTES_PER_PATIENT])[:MAX_CONTEXT_CHARS_PER_PATIENT]

    context_df = (
        ctx.groupby("_pid")["piece"]
        .apply(combine)
        .reset_index()
        .rename(columns={"piece": "clinical_context"})
    )

    return context_df


eval_pids = set(eval_df["_pid"].astype(str))

if retrieved_context is not None:
    print("Using retrieved_context.parquet as judge evidence.")
    context_df = build_context_from_retrieved_context(retrieved_context, eval_pids)
else:
    print("retrieved_context.parquet not found. Building compact clinical context from notes.parquet.")
    context_df = build_context_from_notes(notes, eval_pids)

print("context_df:", context_df.shape)
display(context_df.head())

eval_df = eval_df.merge(context_df, on="_pid", how="left")
eval_df["clinical_context"] = eval_df["clinical_context"].fillna("").astype(str)

print("eval_df after context merge:", eval_df.shape)
display(eval_df.head())

retrieved_context.parquet not found. Building compact clinical context from notes.parquet.
context_df: (100, 2)


,_pid,clinical_context
0,1084,[Physician — 2198-08-02]\nChief Complaint: Pr...
1,15057,[Physician — 2127-05-20]\nChief Complaint:\n ...
2,16076,[Radiology — 2180-10-31]\n[**2180-10-31**] 6:1...
3,20804,[Radiology — 2169-06-05]\n[**2169-6-5**] 1:00 ...
4,21706,[Radiology — 2115-12-25]\n[**2115-12-25**] 8:3...


eval_df after context merge: (100, 7)


,_pid,baseline_summary,rag_summary,rag2_multi_query_summary,rag3_matched_budget_summary,human_reference,clinical_context
0,1084,The patient is a 61-year-old male with a histo...,"The patient, a 61-year-old male with a history...","The patient, a 61-year-old male with a history...","The patient, a 61-year-old male with a history...",Pt is a 61 y.o male with h.o prostate ca with ...,[Physician — 2198-08-02]\nChief Complaint: Pr...
1,4954,**Hospital Course Summary:**\n\nThe patient is...,"The patient, a 64-year-old individual with a h...",The patient is a 63-year-old woman with a hist...,"**Brief Hospital Course:**\n\nThe patient, a 6...","64yo woman with multiple myeloma, s/p allogene...",[Radiology — 2152-03-09]\n[**2152-3-9**] 3:46 ...
2,5954,"The patient, a 56-year-old male with a history...","The patient, a 56-year-old male with a history...",Title: Pacemaker Malfunction and Revision\n\nT...,Title: Pacemaker Malfunction and Revision\n\nA...,Mr. [**Known lastname 46**] is a 56 year-old m...,[Radiology — 2112-02-29]\n[**2112-2-29**] 7:22...
3,6214,**Hospital Course Summary:**\n\nThe patient is...,"The patient, a 73-year-old male, was admitted ...","The patient, a 73-year-old male, was transferr...",The patient is a 73-year-old male with a histo...,The patient was transferred from an OSH. He ar...,[Radiology — 2131-02-21]\n[**2131-2-21**] 8:57...
4,8501,The patient is a 25-year-old woman with a hist...,The patient is a 25-year-old female with a his...,The patient is a 25-year-old female with a his...,"The patient, a 25-year-old female with a histo...",Ms. [**Known lastname **] is a 25yo female wit...,[Radiology — 2146-10-31]\n[**2146-10-31**] 10:...


In [ ]:
# 9. Preflight checks
required_cols = ["_pid", "baseline_summary", "rag_summary", "rag2_multi_query_summary", "rag3_matched_budget_summary", "human_reference", "clinical_context"]
missing_cols = [c for c in required_cols if c not in eval_df.columns]
print("Missing columns:", missing_cols)
if missing_cols:
    print(eval_df.columns.tolist())
    raise ValueError(f"Missing required columns: {missing_cols}")

print("Number of patients:", len(eval_df))
print("Non-empty summaries:")
for col in ["baseline_summary", "rag_summary", "rag2_multi_query_summary", "rag3_matched_budget_summary"]:
    print(col, (eval_df[col].fillna("").astype(str).str.strip() != "").sum())

Missing columns: []
Number of patients: 100
Non-empty summaries:
baseline_summary 100
rag_summary 100
rag2_multi_query_summary 100
rag3_matched_budget_summary 100


In [ ]:
# 10. Load judge model: Qwen/Qwen3-32B
JUDGE_MODEL_NAME = "Qwen/Qwen3-32B"
LOAD_IN_4BIT = True

print("Judge model:", JUDGE_MODEL_NAME)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))

judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL_NAME, trust_remote_code=True)
if LOAD_IN_4BIT:
    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4")
    judge_model = AutoModelForCausalLM.from_pretrained(JUDGE_MODEL_NAME, quantization_config=bnb_config, device_map="auto", trust_remote_code=True)
else:
    judge_model = AutoModelForCausalLM.from_pretrained(JUDGE_MODEL_NAME, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32, device_map="auto", trust_remote_code=True)

if judge_tokenizer.pad_token is None:
    judge_tokenizer.pad_token = judge_tokenizer.eos_token
print("Judge model loaded.")

Judge model: Qwen/Qwen3-32B
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Judge model loaded.


In [ ]:
# 11. JSON extraction, model call, and repair

def remove_thinking_blocks(text):
    if text is None: return ""
    text = str(text)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL | re.IGNORECASE)
    return text.replace("<think>", "").replace("</think>", "").strip()

def extract_json(text):
    if text is None: return None
    text = remove_thinking_blocks(text).strip().replace("```json", "").replace("```JSON", "").replace("```", "").strip()
    try: return json.loads(text)
    except Exception: pass
    start, end = text.find("{"), text.rfind("}")
    if start != -1 and end != -1 and end > start:
        try: return json.loads(text[start:end+1].strip())
        except Exception: pass
    return None

def run_judge(prompt, model, tokenizer, max_new_tokens=500):
    messages = [
        {"role":"system", "content":"You are a strict clinical summary evaluator. Output exactly one valid JSON object only. No markdown. No thinking steps. No explanations outside JSON."},
        {"role":"user", "content":prompt},
    ]
    try:
        formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt", truncation=True, max_length=10000).to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id, eos_token_id=tokenizer.eos_token_id)
    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    raw = remove_thinking_blocks(raw)
    return raw, extract_json(raw)

def repair_to_json(raw_text, model, tokenizer, max_new_tokens=500):
    repair_prompt = f'''
Convert the evaluator output below into exactly one valid JSON object.
Return JSON only. No markdown. No thinking steps.

Use this schema:
{{
  "summary_name": "string",
  "factuality_score": integer,
  "hallucination_score": integer,
  "completeness_score": integer,
  "conciseness_score": integer,
  "overall_quality_score": integer,
  "major_supported_points": ["string"],
  "major_unsupported_or_hallucinated_points": ["string"],
  "major_missing_points": ["string"],
  "brief_rationale": "string"
}}

Rules: scores are integers from 1 to 5. If unclear, use 3.

Evaluator output:
---
{str(raw_text)[:5000]}
---
Return JSON now:
'''
    return run_judge(repair_prompt, model, tokenizer, max_new_tokens=max_new_tokens)

def safe_run_judge(prompt, model, tokenizer, summary_name="unknown", max_new_tokens=500):
    raw, parsed = run_judge(prompt, model, tokenizer, max_new_tokens=max_new_tokens)
    if parsed is not None:
        parsed["parse_error"] = False
        parsed["raw_output"] = raw
        return raw, parsed
    repaired_raw, repaired = repair_to_json(raw, model, tokenizer, max_new_tokens=max_new_tokens)
    if repaired is not None:
        repaired["parse_error"] = False
        repaired["raw_output"] = raw
        repaired["repair_raw_output"] = repaired_raw
        return raw, repaired
    return raw, {
        "summary_name": summary_name,
        "factuality_score": np.nan,
        "hallucination_score": np.nan,
        "completeness_score": np.nan,
        "conciseness_score": np.nan,
        "overall_quality_score": np.nan,
        "major_supported_points": [],
        "major_unsupported_or_hallucinated_points": [],
        "major_missing_points": [],
        "brief_rationale": "JSON parsing failed after repair.",
        "parse_error": True,
        "raw_output": raw,
        "repair_raw_output": repaired_raw,
    }

In [ ]:
# 12. Strict single-summary prompt

def build_single_summary_prompt(summary_name, summary_text, reference_text, clinical_context):
    summary_name = str(summary_name)
    summary_text = "" if summary_text is None else str(summary_text)
    reference_text = "" if reference_text is None else str(reference_text)
    clinical_context = "" if clinical_context is None else str(clinical_context)
    return f'''
You are evaluating one generated Brief Hospital Course for an ICU patient.

Return exactly one valid JSON object. No markdown. No thinking steps. No repeated text.

Evidence hierarchy:
1. Clinical context is the primary source of truth.
2. Human reference helps identify important missing hospital-course events.
3. Do not assume a generated claim is true unless supported by clinical context.
4. If clinical context is insufficient to verify a claim, treat it as unsupported.

Scoring:
- factuality_score: 1 to 5, higher is better.
- hallucination_score: 1 to 5, lower is better.
- completeness_score: 1 to 5, higher is better.
- conciseness_score: 1 to 5, higher is better.
- overall_quality_score: 1 to 5, higher is better.

Calibration:
- Do not give 5 unless nearly perfect.
- Most summaries should receive 2 to 4.
- Generic/vague summary: completeness_score <= 3.
- Missing important diagnoses/treatments/complications/ICU events/procedures/discharge plans: completeness_score <= 3.
- Unsupported claims: factuality_score <= 3 and hallucination_score >= 3.
- Clinically misleading statements: overall_quality_score <= 3.

Return exactly this JSON schema:
{{
  "summary_name": "{summary_name}",
  "factuality_score": 3,
  "hallucination_score": 3,
  "completeness_score": 3,
  "conciseness_score": 3,
  "overall_quality_score": 3,
  "major_supported_points": ["short phrase"],
  "major_unsupported_or_hallucinated_points": ["short phrase"],
  "major_missing_points": ["short phrase"],
  "brief_rationale": "one short paragraph"
}}

Clinical context:
---
{clinical_context[:12000]}
---

Human reference Brief Hospital Course:
---
{reference_text[:4000]}
---

Generated summary name: {summary_name}

Generated summary:
---
{summary_text[:4000]}
---

Return JSON now:
'''

In [ ]:
# 13. Optional one-patient JSON test
# Run this cell first. If PARSED JSON is valid, continue to the full run.
test_row = eval_df.iloc[0]
test_prompt = build_single_summary_prompt(
    summary_name="rag3_matched_budget",
    summary_text=test_row["rag3_matched_budget_summary"],
    reference_text=test_row["human_reference"],
    clinical_context=test_row["clinical_context"],
)
raw, parsed = safe_run_judge(test_prompt, judge_model, judge_tokenizer, summary_name="rag3_matched_budget", max_new_tokens=500)
print("RAW OUTPUT:")
print(raw)
print("PARSED JSON:")
print(parsed)

RAW OUTPUT:
{
  "summary_name": "rag3_matched_budget",
  "factuality_score": 3,
  "hallucination_score": 3,
  "completeness_score": 3,
  "conciseness_score": 3,
  "overall_quality_score": 3,
  "major_supported_points": [
    "Patient presented with AMS",
    "Found altered at home",
    "Received Narcan",
    "Intubated in ED",
    "CT scan and LP were negative",
    "Treated with vancomycin and zosyn",
    "Self-extubated",
    "Mental status improved",
    "Positive urine cultures",
    "Bradycardia noted"
  ],
  "major_unsupported_or_hallucinated_points": [
    "Leukocytosis reported",
    "Elevated creatinine",
    "Positive urine cultures for barbiturates, opiates, and methadone",
    "Subcutaneous heparin administered",
    "Toxic metabolic etiology listed as possible cause",
    "Blood cultures and thyroid function tests planned"
  ],
  "major_missing_points": [
    "Opiate withdrawal as likely cause",
    "Ambien ingestion identified",
    "Antibiotics discontinued after extuba

In [ ]:
# 14. Full all-patient single-summary evaluation with checkpointing
N_EVAL = len(eval_df)
RUN_TAG = "qwen3_32b_jsonfix_with_rag3_single_only_all"
CHECKPOINT_PATH = f"{DATA_DIR}/llm_judge_{RUN_TAG}_checkpoint.parquet"
FINAL_SINGLE_EVAL_PATH = f"{DATA_DIR}/llm_judge_{RUN_TAG}_single_scores.parquet"
SUMMARY_TABLE_PATH = f"{DATA_DIR}/llm_judge_{RUN_TAG}_summary_table.csv"
RAW_JSONL_PATH = f"{DATA_DIR}/llm_judge_{RUN_TAG}_raw_records.jsonl"
SAVE_EVERY = 1

available_systems = [(name, col) for name, col in [
    ("baseline", "baseline_summary"),
    ("rag", "rag_summary"),
    ("rag2_multi_query", "rag2_multi_query_summary"),
    ("rag3_matched_budget", "rag3_matched_budget_summary"),
] if col in eval_df.columns]
print("Available systems:", available_systems)
print("Total patients to evaluate:", N_EVAL)
print("Checkpoint path:", CHECKPOINT_PATH)

if os.path.exists(CHECKPOINT_PATH):
    single_eval_df = pd.read_parquet(CHECKPOINT_PATH)
    single_eval_records = single_eval_df.to_dict("records")
    print("Loaded checkpoint:", CHECKPOINT_PATH, "records:", len(single_eval_records))
else:
    single_eval_records = []
    single_eval_df = pd.DataFrame()
    print("No checkpoint found. Starting from scratch.")

completed_keys = set()
for r in single_eval_records:
    if "_pid" in r and "summary_type" in r:
        completed_keys.add((str(r["_pid"]), str(r["summary_type"])))
print("Completed patient-system evaluations:", len(completed_keys))

Available systems: [('baseline', 'baseline_summary'), ('rag', 'rag_summary'), ('rag2_multi_query', 'rag2_multi_query_summary'), ('rag3_matched_budget', 'rag3_matched_budget_summary')]
Total patients to evaluate: 100
Checkpoint path: /content/drive/MyDrive/BMI702 Project/Data/llm_judge_qwen3_32b_jsonfix_with_rag3_single_only_all_checkpoint.parquet
No checkpoint found. Starting from scratch.
Completed patient-system evaluations: 0


In [ ]:
# 15. Helper functions for records and JSONL checkpoint
def ensure_list(x):
    if isinstance(x, list): return [str(v) for v in x]
    if x is None or (isinstance(x, float) and np.isnan(x)): return []
    return [str(x)]

def normalize_single_eval_record(parsed_json, pid, system_name, raw_output=None):
    if parsed_json is None: parsed_json = {}
    return {
        "_pid": str(pid),
        "summary_type": str(system_name),
        "summary_name": parsed_json.get("summary_name", system_name),
        "factuality_score": parsed_json.get("factuality_score", np.nan),
        "hallucination_score": parsed_json.get("hallucination_score", np.nan),
        "completeness_score": parsed_json.get("completeness_score", np.nan),
        "conciseness_score": parsed_json.get("conciseness_score", np.nan),
        "overall_quality_score": parsed_json.get("overall_quality_score", np.nan),
        "major_supported_points": ensure_list(parsed_json.get("major_supported_points", [])),
        "major_unsupported_or_hallucinated_points": ensure_list(parsed_json.get("major_unsupported_or_hallucinated_points", [])),
        "major_missing_points": ensure_list(parsed_json.get("major_missing_points", [])),
        "brief_rationale": str(parsed_json.get("brief_rationale", "")),
        "parse_error": bool(parsed_json.get("parse_error", False)),
        "raw_output": parsed_json.get("raw_output", raw_output),
        "repair_raw_output": parsed_json.get("repair_raw_output", None),
    }

def append_jsonl(path, record):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False, default=str) + "")

In [ ]:
# 16. Main loop
eval_subset = eval_df.head(N_EVAL).copy()
patients_completed_since_save = 0

for _, row in tqdm(eval_subset.iterrows(), total=len(eval_subset), desc="Patients"):
    pid = str(row["_pid"])
    reference_text = row.get("human_reference", "")
    clinical_context = row.get("clinical_context", "")

    for system_name, summary_col in available_systems:
        key = (pid, system_name)
        if key in completed_keys:
            continue

        summary_text = str(row.get(summary_col, "")).strip()
        if len(summary_text) == 0 or summary_text.lower() == "nan":
            record = {
                "_pid": pid, "summary_type": system_name, "summary_name": system_name,
                "factuality_score": np.nan, "hallucination_score": np.nan, "completeness_score": np.nan,
                "conciseness_score": np.nan, "overall_quality_score": np.nan,
                "major_supported_points": [], "major_unsupported_or_hallucinated_points": [], "major_missing_points": [],
                "brief_rationale": "Empty or missing summary.", "parse_error": True,
                "raw_output": None, "repair_raw_output": None,
            }
            single_eval_records.append(record)
            completed_keys.add(key)
            append_jsonl(RAW_JSONL_PATH, record)
            continue

        prompt = build_single_summary_prompt(system_name, summary_text, reference_text, clinical_context)
        raw_output, parsed_json = safe_run_judge(prompt, judge_model, judge_tokenizer, summary_name=system_name, max_new_tokens=500)
        record = normalize_single_eval_record(parsed_json, pid, system_name, raw_output)
        single_eval_records.append(record)
        completed_keys.add(key)
        append_jsonl(RAW_JSONL_PATH, record)

    patients_completed_since_save += 1
    if patients_completed_since_save >= SAVE_EVERY:
        single_eval_df = pd.DataFrame(single_eval_records)
        single_eval_df.to_parquet(CHECKPOINT_PATH, index=False)
        print(f"Checkpoint saved: {len(single_eval_df)} records")
        patients_completed_since_save = 0

single_eval_df = pd.DataFrame(single_eval_records)
single_eval_df.to_parquet(CHECKPOINT_PATH, index=False)
print("Single-summary evaluation completed.")
print("Total records:", len(single_eval_df))
display(single_eval_df.head(20))

Patients:   0%|          | 0/100 [00:00<?, ?it/s]

Checkpoint saved: 4 records
Checkpoint saved: 8 records
Checkpoint saved: 12 records
Checkpoint saved: 16 records
Checkpoint saved: 20 records
Checkpoint saved: 24 records
Checkpoint saved: 28 records
Checkpoint saved: 32 records
Checkpoint saved: 36 records
Checkpoint saved: 40 records
Checkpoint saved: 44 records
Checkpoint saved: 48 records
Checkpoint saved: 52 records
Checkpoint saved: 56 records
Checkpoint saved: 60 records
Checkpoint saved: 64 records
Checkpoint saved: 68 records
Checkpoint saved: 72 records
Checkpoint saved: 76 records
Checkpoint saved: 80 records
Checkpoint saved: 84 records
Checkpoint saved: 88 records
Checkpoint saved: 92 records
Checkpoint saved: 96 records
Checkpoint saved: 100 records
Checkpoint saved: 104 records
Checkpoint saved: 108 records
Checkpoint saved: 112 records
Checkpoint saved: 116 records
Checkpoint saved: 120 records
Checkpoint saved: 124 records
Checkpoint saved: 128 records
Checkpoint saved: 132 records
Checkpoint saved: 136 records
Checkp

,_pid,summary_type,summary_name,factuality_score,hallucination_score,completeness_score,conciseness_score,overall_quality_score,major_supported_points,major_unsupported_or_hallucinated_points,major_missing_points,brief_rationale,parse_error,raw_output,repair_raw_output
0,1084,baseline,baseline,3.0,3.0,3.0,3.0,3.0,"[Patient found altered at home, Intubated due ...","[Heart rate of 171/74, Received haldol and ben...","[Self-extubation, Switch between propofol and ...",The summary includes some accurate details abo...,False,"{\n ""summary_name"": ""baseline"",\n ""factualit...",NaN
1,1084,rag,rag,2.0,3.0,2.0,3.0,2.0,[Patient presented with AMS after being found ...,"[Leukocytosis reported, Positive urine tests f...","[Patient was bradycardic during ICU stay, Swit...",The summary contains several unsupported claim...,False,"{\n ""summary_name"": ""rag"",\n ""factuality_sco...",NaN
2,1084,rag2_multi_query,rag2_multi_query,3.0,3.0,3.0,3.0,3.0,"[Patient found confused and naked at home, Int...","[Positive urine tests for barbiturates, methad...","[Self-extubation event, Switching from propofo...",The summary captures key events like intubatio...,False,"{\n ""summary_name"": ""rag2_multi_query"",\n ""f...",NaN
3,1084,rag3_matched_budget,rag3_matched_budget,3.0,3.0,3.0,3.0,3.0,"[Patient presented with AMS, Found altered at ...","[Leukocytosis reported, Elevated creatinine, P...","[Opiate withdrawal as likely cause, Ambien ing...",The summary captures key events like AMS prese...,False,"{\n ""summary_name"": ""rag3_matched_budget"",\n ...",NaN
4,4954,baseline,baseline,3.0,3.0,2.0,3.0,2.0,"[Patient has multiple myeloma and amyloidosis,...","[Day-by-day progression from Day 1 to Day 9, D...","[Amyloidosis involving the heart, End-stage re...",The summary accurately captures some key clini...,False,"{\n ""summary_name"": ""baseline"",\n ""factualit...",NaN
5,4954,rag,rag,2.0,3.0,2.0,3.0,2.0,"[Patient has multiple myeloma and amyloidosis,...","[Patient was discharged home after a week, Pat...","[Amyloidosis affecting the heart, Hypercalcemi...",The summary contains several unsupported claim...,False,"{\n ""summary_name"": ""rag"",\n ""factuality_sco...",NaN
6,4954,rag2_multi_query,rag2_multi_query,3.0,3.0,3.0,3.0,3.0,"[Patient has amyloidosis and multiple myeloma,...",[Persistent right middle lobe and right lower ...,"[Amyloidosis involving the heart, End-stage re...",The summary includes some accurate points abou...,False,"{\n ""summary_name"": ""rag2_multi_query"",\n ""f...",NaN
7,4954,rag3_matched_budget,rag3_matched_budget,3.0,3.0,3.0,3.0,3.0,"[Patient has multiple myeloma and amyloidosis,...","[Chest tube removed on March 12, Transferred t...","[Hypercalcemia management, Supratherapeutic IN...",The summary accurately reflects some key clini...,False,"{\n ""summary_name"": ""rag3_matched_budget"",\n ...",NaN
8,5954,baseline,baseline,3.0,3.0,3.0,4.0,3.0,"[Patient has atrial fibrillation, complete hea...","[Patient underwent lead revision on 3/1/2021, ...","[New right-sided pacemaker implanted, Explante...",The summary accurately reflects the patient's ...,False,"{\n ""summary_name"": ""baseline"",\n ""factualit...",NaN
9,5954,rag,rag,3.0,3.0,3.0,3.0,3.0,[Patient transferred due to pacemaker malfunct...,"[Potassium was replenished, Patient was kept N...","[Patient had junctional rhythm at low 40s, Vit...",The summary correctly captures the patient's t...,False,"{\n ""summary_name"": ""rag"",\n ""factuality_sco...",NaN


In [ ]:
# 17. Clean scores and summarize results
score_cols = ["factuality_score", "hallucination_score", "completeness_score", "conciseness_score", "overall_quality_score"]
for col in score_cols:
    if col not in single_eval_df.columns: single_eval_df[col] = np.nan
    single_eval_df[col] = pd.to_numeric(single_eval_df[col], errors="coerce")
if "parse_error" not in single_eval_df.columns: single_eval_df["parse_error"] = False

parse_error_table = single_eval_df.groupby("summary_type")["parse_error"].sum().reset_index()
print("Parse error counts by system:")
display(parse_error_table)

valid_counts = single_eval_df.groupby("summary_type")["overall_quality_score"].count().reset_index(name="valid_overall_quality_count")
print("Valid score counts by system:")
display(valid_counts)

summary_table = single_eval_df.groupby("summary_type")[score_cols].agg(["mean", "std", "count"]).reset_index()
print("All-patient mean judge score table:")
display(summary_table)

Parse error counts by system:


,summary_type,parse_error
0,baseline,5
1,rag,5
2,rag2_multi_query,3
3,rag3_matched_budget,1


Valid score counts by system:


,summary_type,valid_overall_quality_count
0,baseline,95
1,rag,95
2,rag2_multi_query,97
3,rag3_matched_budget,99


All-patient mean judge score table:


summary_type factuality_score                 hallucination_score  \
                                   mean       std count                mean   
0             baseline         2.915789  0.314986    95            2.989474   
1                  rag         2.694737  0.462962    95            2.947368   
2     rag2_multi_query         2.804124  0.398935    97            2.958763   
3  rag3_matched_budget         2.919192  0.340371    99            2.969697   

                  completeness_score                 conciseness_score  \
        std count               mean       std count              mean   
0  0.102598    95           2.642105  0.481924    95          3.389474   
1  0.224481    95           2.536842  0.501286    95          3.115789   
2  0.199871    97           2.649485  0.479610    97          3.175258   
3  0.172292    99           2.797980  0.403551    99          3.161616   

                  overall_quality_score                  
        std count                  mean       std count  
0  0.490218    95              2.642105  0.481924    95  
1  0.321670    95              2.536842  0.501286    95  
2  0.382162    97              2.649485  0.479610    97  
3  0.369972    99              2.797980  0.403551    99

In [ ]:
# 18. Save final outputs
single_eval_df.to_parquet(FINAL_SINGLE_EVAL_PATH, index=False)
summary_table.to_csv(SUMMARY_TABLE_PATH, index=False)

summary_table_flat = summary_table.copy()
summary_table_flat.columns = ["_".join([str(x) for x in col if str(x) != ""]).strip("_") if isinstance(col, tuple) else str(col) for col in summary_table_flat.columns]
REPORT_SUMMARY_CSV = f"{DATA_DIR}/llm_judge_{RUN_TAG}_summary_table_flat.csv"
summary_table_flat.to_csv(REPORT_SUMMARY_CSV, index=False)

print("Saved final outputs:")
print("Single-summary scores:", FINAL_SINGLE_EVAL_PATH)
print("Summary table:", SUMMARY_TABLE_PATH)
print("Flat summary table:", REPORT_SUMMARY_CSV)
print("Checkpoint:", CHECKPOINT_PATH)
print("Raw JSONL records:", RAW_JSONL_PATH)
display(summary_table_flat)

Saved final outputs:
Single-summary scores: /content/drive/MyDrive/BMI702 Project/Data/llm_judge_qwen3_32b_jsonfix_with_rag3_single_only_all_single_scores.parquet
Summary table: /content/drive/MyDrive/BMI702 Project/Data/llm_judge_qwen3_32b_jsonfix_with_rag3_single_only_all_summary_table.csv
Flat summary table: /content/drive/MyDrive/BMI702 Project/Data/llm_judge_qwen3_32b_jsonfix_with_rag3_single_only_all_summary_table_flat.csv
Checkpoint: /content/drive/MyDrive/BMI702 Project/Data/llm_judge_qwen3_32b_jsonfix_with_rag3_single_only_all_checkpoint.parquet
Raw JSONL records: /content/drive/MyDrive/BMI702 Project/Data/llm_judge_qwen3_32b_jsonfix_with_rag3_single_only_all_raw_records.jsonl


,summary_type,factuality_score_mean,factuality_score_std,factuality_score_count,hallucination_score_mean,hallucination_score_std,hallucination_score_count,completeness_score_mean,completeness_score_std,completeness_score_count,conciseness_score_mean,conciseness_score_std,conciseness_score_count,overall_quality_score_mean,overall_quality_score_std,overall_quality_score_count
0,baseline,2.915789,0.314986,95,2.989474,0.102598,95,2.642105,0.481924,95,3.389474,0.490218,95,2.642105,0.481924,95
1,rag,2.694737,0.462962,95,2.947368,0.224481,95,2.536842,0.501286,95,3.115789,0.321670,95,2.536842,0.501286,95
2,rag2_multi_query,2.804124,0.398935,97,2.958763,0.199871,97,2.649485,0.479610,97,3.175258,0.382162,97,2.649485,0.479610,97
3,rag3_matched_budget,2.919192,0.340371,99,2.969697,0.172292,99,2.797980,0.403551,99,3.161616,0.369972,99,2.797980,0.403551,99
